# YOLO26x-seg для папки `Картинки`

Notebook запускает только сегментацию колоний:

1. Исходное изображение resize до 736x736 без поиска чашки.
2. YOLO26x-seg instance segmentation колоний.

Детектор чашек, anomaly detection, извлечение признаков, scoring, selected/review candidates и stability-фильтры здесь не запускаются.

In [ ]:
from pathlib import Path
import json
import re
import sys

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPyImage
from tqdm.auto import tqdm
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "full_pipline":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from full_pipline.full_pipeline import (
    collect_image_paths,
    make_full_pipeline_config,
    normalize_masks_input,
    predict_colony_masks,
    prepare_image_for_segmentation,
    read_image_rgb,
    smooth_masks_before_anomaly,
    build_mask_smoothing_diagnostics,
)

print("Корень проекта:", PROJECT_ROOT)

## Настройки

In [ ]:
INPUT_DIR = Path(r"C:/ColonyNet/Картинки")
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "colony_segmentation_only_kartinki_pipeline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_PIPELINE = True
SAVE_OVERLAYS = True
SAVE_LABELED_MASKS = True
DISPLAY_ALL_MASK_VISUALIZATIONS = True
VISUALIZATION_GRID_COLS = 3
VISUALIZATION_BATCH_SIZE = 12

config = make_full_pipeline_config(output_dir=OUTPUT_DIR)

# Только inference YOLO26x-seg. Детектор чашек отключен: весь кадр resize до 736x736.
config.preprocess_mode = "resize_only"
config.use_petri_detector = False
config.save_visualizations = False
config.save_csv = False
config.save_xlsx = False
config.save_colony_crops = False
config.save_feature_space_plot = False
config.save_feature_correlation_report = False

print("Input dir:", INPUT_DIR)
print("Output dir:", OUTPUT_DIR)
print("Модель сегментации колоний:", config.model_weights_path)
print("Сегментатор существует:", Path(config.model_weights_path).exists())

## Вспомогательные функции сохранения и визуализации

In [ ]:
def write_image(path: str | Path, image: np.ndarray, rgb: bool = True) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    arr = np.asarray(image)
    if arr.ndim == 3 and rgb:
        arr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
    ok, encoded = cv2.imencode(path.suffix or ".png", arr)
    if not ok:
        raise IOError(f"Не удалось закодировать изображение: {path}")
    encoded.tofile(str(path))
    return path


def safe_output_name(image_path: Path, root: Path) -> str:
    try:
        rel = image_path.relative_to(root).with_suffix("")
        name = "__".join(rel.parts)
    except ValueError:
        name = image_path.stem
    return re.sub(r'[<>:"/\\|?*]+', "_", name).strip(" .") or "image"


def masks_to_labeled_image(masks, image_shape=None) -> np.ndarray:
    normalized = normalize_masks_input(masks)
    if not normalized:
        if image_shape is None:
            return np.zeros((0, 0), dtype=np.uint16)
        h, w = [int(v) for v in image_shape[:2]]
        return np.zeros((h, w), dtype=np.uint16)
    h, w = normalized[0].shape
    labeled = np.zeros((h, w), dtype=np.uint16)
    for idx, mask in enumerate(normalized, start=1):
        labeled[np.asarray(mask).astype(bool)] = idx
    return labeled


def color_for_id(idx: int) -> tuple[int, int, int]:
    rng = np.random.default_rng(idx * 1009 + 17)
    return tuple(int(v) for v in rng.integers(40, 255, size=3))


def overlay_instance_masks(image_rgb: np.ndarray, masks) -> np.ndarray:
    view = image_rgb.copy()
    masks = normalize_masks_input(masks)
    if not masks:
        return view

    overlay = view.copy()
    for idx, mask in enumerate(masks, start=1):
        mask_bool = np.asarray(mask).astype(bool)
        color = np.array(color_for_id(idx), dtype=np.uint8)
        overlay[mask_bool] = (0.55 * overlay[mask_bool] + 0.45 * color).astype(np.uint8)

        contours, _ = cv2.findContours(mask_bool.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, contours, -1, tuple(int(v) for v in color.tolist()), 1)

    return overlay


def show_images(images, titles=None, cols=2, figsize=(12, 8), save_path=None):
    if not images:
        return
    cols = max(1, int(cols))
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.set_axis_off()
    for idx, image in enumerate(images):
        axes[idx].imshow(image)
        if titles:
            axes[idx].set_title(titles[idx], fontsize=10)
    fig.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=160, bbox_inches="tight", pad_inches=0.1)
    display(fig)
    plt.close(fig)

## Список изображений

In [ ]:
image_paths = collect_image_paths(INPUT_DIR)
print("Найдено изображений:", len(image_paths))
display(pd.DataFrame({"image_name": [p.name for p in image_paths], "path": [str(p) for p in image_paths]}))

## Запуск: detector чашки -> crop 736 -> segmentation колоний

In [ ]:
results = {}
summary_rows = []
detection_tables = []

if RUN_PIPELINE:
    segmentation_weights_path = Path(config.model_weights_path)
    if not segmentation_weights_path.exists():
        raise FileNotFoundError(f"Segmentation weights file not found: {segmentation_weights_path}")
    print("Segmentation weights:", segmentation_weights_path)
    segmentation_model = YOLO(str(segmentation_weights_path))

    for image_path in tqdm(image_paths, desc="YOLO26x-seg"):
        out_name = safe_output_name(image_path, INPUT_DIR)
        image_output_dir = OUTPUT_DIR / out_name
        image_output_dir.mkdir(parents=True, exist_ok=True)

        row = {
            "image_name": image_path.name,
            "input_path": str(image_path),
            "output_dir": str(image_output_dir),
            "status": "ok",
            "error": "",
        }

        try:
            image_original = read_image_rgb(image_path)
            prepared_image, preprocess = prepare_image_for_segmentation(
                image_original,
                detector_model=None,
                config=config,
            )
            masks, detections_df = predict_colony_masks(
                model=segmentation_model,
                image_rgb=prepared_image,
                config=config,
            )

            masks_yolo = masks
            apply_mask_smoothing = bool(getattr(config, "smooth_masks_before_anomaly", True))
            mask_smoothing_sigma = float(getattr(config, "mask_smoothing_sigma", 1.6))
            mask_smoothing_threshold = float(getattr(config, "mask_smoothing_threshold", 0.50))
            if apply_mask_smoothing:
                masks, smoothing_df = smooth_masks_before_anomaly(
                    masks_yolo,
                    config=config,
                    sigma=mask_smoothing_sigma,
                    threshold=mask_smoothing_threshold,
                )
            else:
                masks = normalize_masks_input(masks_yolo)
                smoothing_df = build_mask_smoothing_diagnostics(
                    masks,
                    sigma=mask_smoothing_sigma,
                    threshold=mask_smoothing_threshold,
                    applied=False,
                )

            detections_df = detections_df.copy()
            if not detections_df.empty and not smoothing_df.empty and "colony_id" in detections_df.columns:
                detections_df = detections_df.merge(smoothing_df, on="colony_id", how="left")
            detections_df.insert(0, "image_name", image_path.name)
            detections_df.to_csv(image_output_dir / "detections_yolo.csv", index=False, encoding="utf-8-sig")
            smoothing_df.to_csv(image_output_dir / "mask_smoothing_diagnostics.csv", index=False, encoding="utf-8-sig")
            detection_tables.append(detections_df)

            preprocess = dict(preprocess)
            preprocess["mask_smoothing_before_anomaly"] = apply_mask_smoothing
            preprocess["mask_smoothing_sigma"] = mask_smoothing_sigma
            preprocess["mask_smoothing_threshold"] = mask_smoothing_threshold

            with open(image_output_dir / "preprocess.json", "w", encoding="utf-8") as f:
                json.dump(preprocess, f, ensure_ascii=False, indent=2)

            if SAVE_OVERLAYS:
                write_image(image_output_dir / "01_input_resized_736.png", prepared_image)
                write_image(image_output_dir / "02_colony_masks_overlay.png", overlay_instance_masks(prepared_image, masks))

            labeled_mask = masks_to_labeled_image(masks, prepared_image.shape[:2])
            if SAVE_LABELED_MASKS:
                np.save(image_output_dir / "masks_labeled.npy", labeled_mask)
                write_image(image_output_dir / "masks_labeled.png", labeled_mask, rgb=False)

            row.update({
                "original_h": image_original.shape[0],
                "original_w": image_original.shape[1],
                "prepared_h": prepared_image.shape[0],
                "prepared_w": prepared_image.shape[1],
                "preprocess_mode": preprocess.get("preprocess_mode", "resize_only"),
                "crop_xyxy": json.dumps(preprocess.get("crop_xyxy", []), ensure_ascii=False),
                "crop_shape": json.dumps(preprocess.get("crop_shape", []), ensure_ascii=False),
                "n_colonies": len(masks),
                "n_yolo_detections": len(detections_df),
            })

            results[image_path.name] = {
                "image_original": image_original,
                "prepared_image": prepared_image,
                "preprocess": preprocess,
                "masks_raw_yolo": masks_yolo,
                "masks": masks,
                "mask_smoothing": smoothing_df,
                "detections": detections_df,
                "output_dir": image_output_dir,
            }

        except Exception as exc:
            row["status"] = "error"
            row["error"] = str(exc)

        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    all_detections_df = pd.concat(detection_tables, ignore_index=True) if detection_tables else pd.DataFrame()
    counts_df = summary_df[[
        "image_name",
        "preprocess_mode",
        "n_colonies",
        "n_yolo_detections",
        "status",
        "error",
    ]].copy()

    summary_df.to_csv(OUTPUT_DIR / "pipeline_summary.csv", index=False, encoding="utf-8-sig")
    counts_df.to_csv(OUTPUT_DIR / "colony_counts_by_image.csv", index=False, encoding="utf-8-sig")
    all_detections_df.to_csv(OUTPUT_DIR / "all_detections_yolo.csv", index=False, encoding="utf-8-sig")

    print("Готово. Output dir:", OUTPUT_DIR)
    print("Всего изображений:", len(counts_df))
    print("Всего колоний:", int(counts_df["n_colonies"].fillna(0).sum()))
    display(counts_df)
    display(summary_df)
else:
    print("RUN_PIPELINE = False. Установите True, чтобы запустить YOLO26x-seg.")

## Общая таблица YOLO detections

In [ ]:
if "all_detections_df" in globals() and not all_detections_df.empty:
    print("Всего colony detections:", len(all_detections_df))
    display(all_detections_df.head(30))
else:
    print("Детекций колоний нет или pipeline еще не запускался.")

## Визуализация всех изображений с масками колоний

In [ ]:
if results and DISPLAY_ALL_MASK_VISUALIZATIONS:
    visualization_items = list(results.items())
    for page_idx, start in enumerate(range(0, len(visualization_items), VISUALIZATION_BATCH_SIZE), start=1):
        batch = visualization_items[start:start + VISUALIZATION_BATCH_SIZE]
        batch_images = [overlay_instance_masks(item[1]["prepared_image"], item[1]["masks"]) for item in batch]
        batch_titles = [f"{name}: {len(data['masks'])} colonies" for name, data in batch]
        show_images(
            batch_images,
            batch_titles,
            cols=VISUALIZATION_GRID_COLS,
            figsize=(4 * VISUALIZATION_GRID_COLS, 4 * int(np.ceil(len(batch) / VISUALIZATION_GRID_COLS))),
            save_path=OUTPUT_DIR / f"all_colony_masks_overlay_page_{page_idx:02d}.png",
        )
elif results:
    print("DISPLAY_ALL_MASK_VISUALIZATIONS = False")
else:
    print("Нет результатов для визуализации.")

## Что сохраняется

В `outputs/two_yolo_kartinki_pipeline` сохраняются:

- `pipeline_summary.csv` - одна строка на изображение.
- `colony_counts_by_image.csv` - количество колоний по каждому изображению.
- `all_detections_yolo.csv` - все YOLO detections колоний из всех изображений.
- `all_colony_masks_overlay_page_*.png` - сводные grid-визуализации всех изображений с масками.
- Для каждого изображения отдельная папка с `preprocess.json`, `detections_yolo.csv`, `01_input_resized_736.png`, `02_colony_masks_overlay.png`, `masks_labeled.npy` и `masks_labeled.png`.

Anomaly-файлы (`colony_anomaly_scores.csv`, `selected_anomalies.csv`, `review_candidates.csv`, `anomalies_visualization.png`) этим notebook не создаются.